In [1]:
import os
import lamini
import torch
import logging
import pandas as pd

In [2]:

from pprint import pprint
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from transformers import TrainingArguments
from llama import BasicModelRunner

c:\Users\Pablo\anaconda3\envs\LLMFineTuning\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Pablo\anaconda3\envs\LLMFineTuning\lib\site-packages\transformers\utils\generic.py:260: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  torch.utils._pytree._register_pytree_node(


In [3]:
import datasets
import jsonlines

In [4]:
logger = logging.getLogger(__name__)
global_config = None

In [5]:
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-70m")

In [6]:
# import pandas as pd
# import datasets
from datasets import Dataset

In [7]:
# finetuning_dataset = Dataset.from_pandas(pd.DataFrame(data=file_root))
# finetuning_dataset = pd.read_excel(file_root, index_col=0)

# Data Prep

In [8]:
def save_qa_to_json(dataframe, filename):
    """
    Selects 'Question' and 'Answer' columns from a DataFrame and saves them to a JSON file.

    Args:
        dataframe (pd.DataFrame): The input DataFrame containing 'Question' and 'Answer' columns.
        filename (str): The name of the JSON file to save the data to.
    """
    qa_df = dataframe[["Question", "Answer"]]
    qa_dict = qa_df.to_dict(orient="records")  # Convert to a list of dictionaries
    with jsonlines.open(filename, 'w') as writer:
      writer.write_all(qa_dict)

In [9]:
column_selection = ["Question", "Answer"]

finetuning_dataset = pd.read_excel(file_root, usecols=column_selection, dtype={'Question': str, "Answer": str} )

NameError: name 'file_root' is not defined

In [ ]:
# Example usage:
save_qa_to_json(finetuning_dataset, "/content/gdrive/MyDrive/LLM/edskbtotal_proc_prepared_new.jsonl")

# Model Fine Tuning

In [10]:
model_name = "EleutherAI/pythia-70m"

dataset_name = "edskbtotal_proc_prepared_new.jsonl"
# dataset_name = "lamini_docs.jsonl"
dataset_path = f"Data/{dataset_name}"
use_hf = False

In [11]:
training_config = {
    "model": {
        "pretrained_name": model_name,
        "max_length" : 2048
    },
    "datasets": {
        "use_hf": use_hf,
        "path": dataset_path
    },
    "verbose": True
}

In [12]:
import utilities as ut

In [13]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
train_dataset, test_dataset = ut.tokenize_and_split_data(training_config, tokenizer)

print(train_dataset)
print(test_dataset)

2024-04-19 15:00:04,189 - DEBUG - utilities - Config: datasets.path: Data/edskbtotal_proc_prepared_new.jsonl
datasets.use_hf: false
model.max_length: 2048
model.pretrained_name: EleutherAI/pythia-70m
verbose: true



tokenize False Data/edskbtotal_proc_prepared_new.jsonl


2024-04-19 15:00:05,296 - DEBUG - fsspec.local - open file: C:/Users/Pablo/.cache/huggingface/datasets/json/default-15c2fdedf55198ff/0.0.0/c8d2d9508a2a2067ab02cd118834ecef34c3700d143b31835ec4235bf10109f7/dataset_info.json
2024-04-19 15:00:05,345 - DEBUG - fsspec.local - open file: C:/Users/Pablo/.cache/huggingface/datasets/json/default-15c2fdedf55198ff/0.0.0/c8d2d9508a2a2067ab02cd118834ecef34c3700d143b31835ec4235bf10109f7/dataset_info.json


Dataset({
    features: ['question', 'answer', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 12787
})
Dataset({
    features: ['question', 'answer', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 1421
})


In [14]:
base_model = AutoModelForCausalLM.from_pretrained(model_name)

In [15]:
device_count = torch.cuda.device_count()
if device_count > 0:
    logger.debug("Select GPU device")
    device = torch.device("cuda")
else:
    logger.debug("Select CPU device")
    device = torch.device("cpu")

2024-04-19 15:00:11,880 - DEBUG - __main__ - Select CPU device


In [16]:
base_model.to(device)

GPTNeoXForCausalLM(
  (gpt_neox): GPTNeoXModel(
    (embed_in): Embedding(50304, 512)
    (emb_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-5): 6 x GPTNeoXLayer(
        (input_layernorm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (post_attention_layernorm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (post_attention_dropout): Dropout(p=0.0, inplace=False)
        (post_mlp_dropout): Dropout(p=0.0, inplace=False)
        (attention): GPTNeoXAttention(
          (rotary_emb): GPTNeoXRotaryEmbedding()
          (query_key_value): Linear(in_features=512, out_features=1536, bias=True)
          (dense): Linear(in_features=512, out_features=512, bias=True)
          (attention_dropout): Dropout(p=0.0, inplace=False)
        )
        (mlp): GPTNeoXMLP(
          (dense_h_to_4h): Linear(in_features=512, out_features=2048, bias=True)
          (dense_4h_to_h): Linear(in_features=2048, out_features=512, bias=True)
          (a

## Setup Training

In [54]:
max_steps = 14500

In [55]:
trained_model_name = f"lamini_docs_{max_steps}_steps"
output_dir = trained_model_name

In [56]:
training_args = TrainingArguments(

  # Learning rate
  learning_rate=1.0e-5,

  # Number of training epochs
  num_train_epochs=1,

  # Max steps to train for (each step is a batch of data)
  # Overrides num_train_epochs, if not -1
  max_steps=max_steps,

  # Batch size for training
  per_device_train_batch_size=1,

  # Directory to save model checkpoints
  output_dir=output_dir,

  # Other arguments
  overwrite_output_dir=False, # Overwrite the content of the output directory
  disable_tqdm=False, # Disable progress bars
  eval_steps=120, # Number of update steps between two evaluations
  save_steps=120, # After # steps model is saved
  warmup_steps=1, # Number of warmup steps for learning rate scheduler
  per_device_eval_batch_size=1, # Batch size for evaluation
  evaluation_strategy="steps",
  logging_strategy="steps",
  logging_steps=1,
  optim="adafactor",
  gradient_accumulation_steps = 4,
  gradient_checkpointing=False,

  # Parameters for early stopping
  load_best_model_at_end=True,
  save_total_limit=1,
  metric_for_best_model="eval_loss",
  greater_is_better=False
)

In [57]:
model_flops = (
  base_model.floating_point_ops(
    {
       "input_ids": torch.zeros(
           (1, training_config["model"]["max_length"])
      )
    }
  )
  * training_args.gradient_accumulation_steps
)

print(base_model)
print("Memory footprint", base_model.get_memory_footprint() / 1e9, "GB")
print("Flops", model_flops / 1e9, "GFLOPs")

GPTNeoXForCausalLM(
  (gpt_neox): GPTNeoXModel(
    (embed_in): Embedding(50304, 512)
    (emb_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-5): 6 x GPTNeoXLayer(
        (input_layernorm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (post_attention_layernorm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (post_attention_dropout): Dropout(p=0.0, inplace=False)
        (post_mlp_dropout): Dropout(p=0.0, inplace=False)
        (attention): GPTNeoXAttention(
          (rotary_emb): GPTNeoXRotaryEmbedding()
          (query_key_value): Linear(in_features=512, out_features=1536, bias=True)
          (dense): Linear(in_features=512, out_features=512, bias=True)
          (attention_dropout): Dropout(p=0.0, inplace=False)
        )
        (mlp): GPTNeoXMLP(
          (dense_h_to_4h): Linear(in_features=512, out_features=2048, bias=True)
          (dense_4h_to_h): Linear(in_features=2048, out_features=512, bias=True)
          (a

In [58]:
trainer = ut.Trainer(
    model = base_model,
    model_flops = model_flops,
    total_steps = max_steps,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = test_dataset,
)

c:\Users\Pablo\anaconda3\envs\LLMFineTuning\lib\site-packages\accelerate\accelerator.py:436: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None)
  warnings.warn(


In [59]:
training_output = trainer.train()

  0%|          | 1/14500 [00:02<9:51:03,  2.45s/it]

{'loss': 2.6538, 'learning_rate': 1e-05, 'epoch': 0.0, 'iter_time': 0.0, 'flops': 0.0, 'remaining_time': 0.0}


  0%|          | 2/14500 [00:05<11:20:05,  2.81s/it]

{'loss': 2.1067, 'learning_rate': 9.999310297261881e-06, 'epoch': 0.0, 'iter_time': 3.0726444721221924, 'flops': 714585703706.72, 'remaining_time': 44547.199556827545}


  0%|          | 3/14500 [00:08<11:40:12,  2.90s/it]

...

 14%|█▍        | 2101/14500 [4:13:55<14:06:34,  4.10s/it]

{'loss': 0.4936, 'learning_rate': 8.551624249948273e-06, 'epoch': 0.66, 'iter_time': 7.253790245510283, 'flops': 302692487380.7874, 'remaining_time': 89939.745254082}


 14%|█▍        | 2102/14500 [4:13:58<13:05:31,  3.80s/it]

{'loss': 0.4015, 'learning_rate': 8.550934547210153e-06, 'epoch': 0.66, 'iter_time': 7.251811942391484, 'flops': 302775062259.5321, 'remaining_time': 89907.96446176962}


KeyboardInterrupt: 

In [25]:
save_dir = f'{output_dir}/final'

trainer.save_model(save_dir)
print("Saved model to:", save_dir)

Saved model to: lamini_docs_100_steps/final


# Model Validation

In [30]:
finetuned_slightly_model = AutoModelForCausalLM.from_pretrained(save_dir, local_files_only=True)

In [31]:
finetuned_slightly_model.to(device) 

GPTNeoXForCausalLM(
  (gpt_neox): GPTNeoXModel(
    (embed_in): Embedding(50304, 512)
    (emb_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-5): 6 x GPTNeoXLayer(
        (input_layernorm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (post_attention_layernorm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (post_attention_dropout): Dropout(p=0.0, inplace=False)
        (post_mlp_dropout): Dropout(p=0.0, inplace=False)
        (attention): GPTNeoXAttention(
          (rotary_emb): GPTNeoXRotaryEmbedding()
          (query_key_value): Linear(in_features=512, out_features=1536, bias=True)
          (dense): Linear(in_features=512, out_features=512, bias=True)
          (attention_dropout): Dropout(p=0.0, inplace=False)
        )
        (mlp): GPTNeoXMLP(
          (dense_h_to_4h): Linear(in_features=512, out_features=2048, bias=True)
          (dense_4h_to_h): Linear(in_features=2048, out_features=512, bias=True)
          (a

In [38]:
import inference as inf

In [52]:
# test_question = "Como me adhiero a icbc club?"
# test_question = "¿Como pido un upgrade de paquete?"
# test_question = "Como puedo obtener un prestamo personal?"
test_question = "Como aviso que voy a viajar para usar la tarjeta afuera?"

# test_question = test_dataset[0]['question']
print("Question input (test):", test_question)

Question input (test): Como aviso que voy a viajar para usar la tarjeta afuera?


In [53]:
print("Finetuned slightly model's answer: ")
print(inf.inference(test_question, finetuned_slightly_model, tokenizer))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Finetuned slightly model's answer: 

<jiero> o que te recomiendo para que te recomiendo para que te recomiendo para que te recomiendo para que te recomiendo para que te recomiendo para que te recomiendo para que te recomiendo para que te recomiendo para que te recomiendo para que te recomiendo para que te recomiendo para que te recomiendo para que te recomiendo para que te recomiendo
